# 🚀 Huấn Luyện AI Gia Sư Tiếng Anh (EnglishAI Tutor) bằng Unsloth & QLoRA

Mô hình được huấn luyện và lưu thành **file nén đặt trực tiếp trong thư mục dự án `AI ENGLISH`**.
Không phụ thuộc Ollama, không cần cài phần mềm ngoài!

⏱️ **Thời gian huấn luyện:** ~15 - 20 phút.
💾 **Đầu ra:** File `english_tutor_model.zip` để lưu trực tiếp vào thư mục dự án của bạn.

### Bước 1: Cài đặt thư viện Unsloth

In [ ]:
!pip install unsloth
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

### Bước 2: Tải Mô hình nền tảng Qwen 2.5 3B (Nén 4-bit siêu nhẹ)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Mô hình nền tảng đã nạp thành công!")

### Bước 3: Cấu hình QLoRA (Fine-Tuning)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ LoRA Adapters đã sẵn sàng!")

### Bước 4: Tải lên tập dữ liệu (dataset_train.jsonl)

In [ ]:
from datasets import load_dataset
from google.colab import files
import os

if not os.path.exists("dataset_train.jsonl"):
    print("Vui lòng chọn file dataset_train.jsonl từ máy tính:")
    uploaded = files.upload()

dataset = load_dataset("json", data_files="dataset_train.jsonl", split="train")

prompt_template = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{instruction}

{input}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

def formatting_prompts_func(examples):
    systems = examples["system"]
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for sys, inst, inp, outp in zip(systems, instructions, inputs, outputs):
        text = prompt_template.format(system=sys, instruction=inst, input=inp, output=outp)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Đã chuẩn bị {len(dataset)} mẫu dữ liệu huấn luyện!")

### Bước 5: Bắt đầu Huấn luyện

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Bắt đầu quá trình huấn luyện...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện thành công!")

### Bước 6: Thử nghiệm chất lượng mô hình vừa train

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """<|im_start|>system
You are Sarah, a warm, patient, and friendly English tutor.<|im_end|>
<|im_start|>user
Can you check this sentence: "She don't likes go to school"<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs)
print(response[0].split("<|im_start|>assistant")[1].replace("<|im_end|>", ""))

### Bước 7: Đóng gói thành 1 file zip để đặt vào thư mục AI ENGLISH

In [ ]:
import os
from google.colab import files

# Lưu trọng số LoRA và Tokenizer đã train
output_dir = "english_tutor_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Đã lưu trọng số vào thư mục {output_dir}")

# Nén thành 1 file duy nhất để lưu trong folder AI ENGLISH
!zip -r english_tutor_model.zip english_tutor_model
print("📦 Đã đóng gói thành file english_tutor_model.zip!")

# Tải trực tiếp về máy tính
files.download("english_tutor_model.zip")